# 05 · Model 2: treatment response, baseline model

**Outcome:** PASI90 at the class-specific window (`configs/treatment_windows.yaml`).
**Predictors:** information available at T0 only (no week-4 data here).
**Models:** null model (always predicts prevalence) and logistic regression.
**Validation:** repeated stratified CV grouped by patient, preprocessing inside each fold.

Goal at this stage: prove the pipeline works end to end without leakage, not to get good results.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import params, path
pd.set_option("display.max_columns", 50)

In [ ]:
# Data source: "fictitious" (test data) or "kruczek_il17" (Polish public dataset)
SOURCE = "fictitious"

if SOURCE == "fictitious":
    from src.data.loaders.fictitious import load
elif SOURCE == "kruczek_il17":
    from src.data.loaders.kruczek_il17 import load

from src.data.clean import clean
raw = load()
tables, cleaning_log = clean(raw)
{k: len(v) for k, v in tables.items()}

## 1. Modelling dataset

In [ ]:
from src.data.cohorts import model2_cohort
from src.outcomes.response import compute_response
from src.features.baseline import model2_features

OUTCOME = params()["model2"]["primary_endpoint"]   # "pasi90"
c2 = model2_cohort(tables)
resp = compute_response(tables, c2)
X_all = model2_features(tables, c2)
d = X_all.merge(resp[["patient_id", "episode_id", OUTCOME]], on=["patient_id", "episode_id"])
d = d.dropna(subset=[OUTCOME]).reset_index(drop=True)
print(f"{len(d)} episodes, {d.patient_id.nunique()} patients, "
      f"{int(d[OUTCOME].sum())} responders / {int((d[OUTCOME] == 0).sum())} non-responders")
d.to_csv(path("processed") / f"model2_dataset_{SOURCE}.csv", index=False)

## 2. Predictors
Defined **before** looking at results. With a small sample, keep the list short.

In [ ]:
NUM = ["age", "sex_male", "bmi", "disease_duration", "pasi_baseline", "bsa_baseline", "dlqi_baseline",
       "difficult_sites", "psoriatic_arthritis", "n_prev_biologic", "prev_failure_inefficacy"]
CAT = ["drug_class"]
# Drop predictors that are constant in this source (e.g. drug_class in a single-class dataset)
NUM = [c for c in NUM if d[c].nunique(dropna=True) > 1]
CAT = [c for c in CAT if d[c].nunique(dropna=True) > 1]
X, y, groups = d[NUM + CAT], d[OUTCOME].astype(int), d.patient_id
print("Predictors:", NUM + CAT)
print(f"Events per predictor: {min(y.sum(), (1 - y).sum()) / len(NUM + CAT):.1f}")

## 3. Cross-validated performance

In [ ]:
from src.models.baseline import null_model, logistic
from src.evaluation.cv import evaluate, summarize, calibration_table

P = params()
models = {"null": null_model(), "logistic": logistic(NUM, CAT, seed=P["seed"])}
results, preds = {}, {}
for name, m in models.items():
    met, pr = evaluate(m, X, y, groups, n_splits=P["cv"]["n_splits"],
                       n_repeats=P["cv"]["n_repeats"], seed=P["seed"])
    results[name], preds[name] = summarize(met), pr
summary = pd.DataFrame(results).T
summary.to_csv(path("tables") / f"model2_baseline_{SOURCE}.csv")
summary

**How to read it:** AUC 0.5 = no better than chance; the null model has AUC ≈ 0.5 by design. Brier: lower is better; the logistic model must beat the null model's Brier to be useful.

## 4. Calibration

In [ ]:
cal = calibration_table(preds["logistic"], n_bins=5)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
ax.plot(cal.predicted, cal.observed, "o-", label="logistic")
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed proportion")
ax.set_title(f"Calibration ({OUTCOME})"); ax.legend()
plt.tight_layout(); plt.savefig(path("figures") / f"calibration_model2_{SOURCE}.png", dpi=200); plt.show()
cal

## 5. Coefficients (model fitted on all data, for interpretation only)

In [ ]:
import numpy as np
m = logistic(NUM, CAT, seed=P["seed"]).fit(X, y)
names = m.named_steps["prep"].get_feature_names_out()
coef = pd.Series(m.named_steps["model"].coef_[0], index=names).sort_values()
coef.round(2)

## 6. Sensitivity analyses (to do)
- Window source `sensitivity` (NICE) instead of `primary` (SmPC): `compute_response(tables, c2, source="sensitivity")`
- PASI75 and DLQI improvement as outcomes
- DLQI MCID = 5